In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots

include("functions.jl")

# ------------------ CONFIG ------------------
Random.seed!(2025)

const bb = 27      # Gamma scale
const aa = 38      # Gamma shape
const N  = 1_462_439
const I0 = 1
const S0 = 1_316_195
const n_iter = 1_000_000

Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83,
             67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]

tau = length(Istar_obs)

# output directory
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

# only two parameters now
header_cont = ["beta", "gamma"]

# ------------------ RUN ONE CHAIN ------------------
c = 3
@info "[no-alarm model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(:none)
t0 = Dates.now()

try
    samples, loglik_aug_vecs = mcmc_one_chain_with_Rstar!(
        Istar_obs, N, S0, I0;
        n_iter=n_iter,
        initθ=initθ_chain
    )

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # --- save samples ---
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # --- save per-time log-likelihoods ---
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))

    el = Dates.value(Dates.now() - t0) / 1000
    @info @sprintf("Chain %d finished in %.2f s -> saved to %s", c, el, out_dir)
catch err
    el = Dates.value(Dates.now() - t0) / 1000
    @warn "Chain $c failed after $el seconds: $err"
end

@info "All chains completed -> output dir: $out_dir"


[ Info: [no-alarm model] Fitting chain 3 (tau=34)
[ Info: [no-alarm] iter 1000/1000000 elapsed=2.9s, rate=0.057, mean=[0.085, 821.007], std=[0.2388, 1.0984] [ADAPT]
[ Info: [no-alarm] iter 2000/1000000 elapsed=4.7s, rate=0.029, mean=[0.043, 820.937], std=[0.1730, 0.7790] [ADAPT]
[ Info: [no-alarm] iter 3000/1000000 elapsed=5.9s, rate=0.019, mean=[0.028, 820.914], std=[0.1424, 0.6367] [ADAPT]
[ Info: [no-alarm] iter 4000/1000000 elapsed=7.2s, rate=0.015, mean=[0.021, 820.903], std=[0.1238, 0.5516] [ADAPT]
[ Info: [no-alarm] iter 5000/1000000 elapsed=8.4s, rate=0.012, mean=[0.017, 820.896], std=[0.1110, 0.4935] [ADAPT]
[ Info: [no-alarm] iter 6000/1000000 elapsed=9.6s, rate=0.010, mean=[0.014, 820.890], std=[0.1015, 0.4507] [ADAPT]
[ Info: [no-alarm] iter 7000/1000000 elapsed=10.8s, rate=0.008, mean=[0.012, 820.887], std=[0.0941, 0.4173] [ADAPT]
[ Info: [no-alarm] iter 8000/1000000 elapsed=12.1s, rate=0.007, mean=[0.011, 820.884], std=[0.0881, 0.3904] [ADAPT]
[ Info: [no-alarm] iter 9000